## Setup and Dependencies

In [28]:
import base64
import json
import os
import time
from pathlib import Path
from typing import Any

from openai import AzureOpenAI
from dotenv import load_dotenv
from PIL import Image

In [33]:
# Load environment variables
load_dotenv()

# Print environment variables for debugging
print("Environment Variables:")
print(f"AZURE_OPENAI_API_KEY: {'*' * 10 if os.getenv('AZURE_OPENAI_API_KEY') else 'Not set'}")
print(f"AZURE_OPENAI_API_VERSION: {os.getenv('AZURE_OPENAI_API_VERSION', '2024-12-01-preview')}")
print(f"AZURE_OPENAI_ENDPOINT: {os.getenv('AZURE_OPENAI_ENDPOINT')}")
print()

# Initialize Azure OpenAI client
client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT") or ""
)

Environment Variables:
AZURE_OPENAI_API_KEY: **********
AZURE_OPENAI_API_VERSION: 2024-12-01-preview
AZURE_OPENAI_ENDPOINT: https://rober-m42gwi28-eastus2.cognitiveservices.azure.com/



## Valid ID - Signature Extraction Prompt

This is the detailed prompt that instructs the vision model to extract signatures from valid ID images.

In [ ]:
SIGNATURE_EXTRACTION_PROMPT_NOCENTER = """
You are an expert computer vision assistant specializing in document analysis and signature detection. Your task is to identify and locate handwritten signatures within images.

## Task
Analyze the provided image and identify all handwritten signatures present. A signature is a person's name written in a distinctive, personalized handwriting style, typically used for authentication or authorization purposes.

## Instructions

1. **Carefully examine the entire image** for handwritten signatures
2. **Distinguish signatures from other handwritten text** - signatures typically have:
   - A more stylized, flowing, or cursive appearance
   - Personal flourishes or unique characteristics
   - Different ink color or pen pressure than printed text
   - Placement in typical signature locations (bottom of documents, signature lines, etc.)

3. **For each signature found**, provide:
   - **Location**: Describe where in the image the signature appears (e.g., "bottom right corner", "below the printed text", "on the signature line")
   - **Bounding box coordinates**: Estimated x, y, width, height as percentages of image dimensions
   - **Confidence level**: High, Medium, or Low
   - **Characteristics**: Brief description of the signature's appearance (e.g., "cursive script in blue ink", "printed signature in black")
   - **Legibility**: Whether the signature is legible enough to read the name
   - **Is owner signature**: Boolean indicating if this signature is by the owner/cardholder of the document
     - Set to `true` if the signature appears to be from the document owner, cardholder, Licensee, or primary subject
     - Set to `false` ONLY if there is clear evidence the signature is from an authorized representative, chairman, witness, guarantor, or other third party
     - Look for explicit labels ("Chairman", "Witness", "Authorized Signatory", "Guardian", etc.)
     - **Default to `true` when ownership is ambiguous or unclear**

4. **Exclude the following**:
   - Printed signatures or typed names
   - Initials alone (unless clearly part of a signature)
   - Random handwritten notes that are not signatures
   - Stamps or seal impressions (unless they contain a handwritten component)

## Output Format

Provide your response in the following JSON structure:

```json
{
  "signatures_found": <number>,
  "signatures": [
    {
      "id": 1,
      "location_description": "<descriptive location>",
      "bounding_box": {
        "x": <percentage>,
        "y": <percentage>,
        "width": <percentage>,
        "height": <percentage>
      },
      "confidence": "<High|Medium|Low>",
      "characteristics": "<description>",
      "legible": <true|false>,
      "estimated_name": "<name if legible, otherwise null>",
      "is_owner_signature": <true|false>
    }
  ],
  "analysis_notes": "<any additional observations or challenges>"
}
```

## Examples of What to Look For

**Typical Signature Characteristics:**
- Flowing, connected cursive writing
- Underlines or flourishes beneath the name
- Rapid, confident strokes
- May be partially illegible due to speed of writing
- Often appears on designated signature lines
- May include date nearby

**Common Locations:**
- Bottom of contracts or forms
- Next to "Signature:" or "Signed by:" labels
- On signature lines (indicated by "___________")
- In signature blocks with printed name underneath
- On checks, receipts, or official documents

**Owner vs Non-Owner Signatures:**
- **Owner signatures** typically appear on:
  - Primary signature line of ID cards or documents
  - "Cardholder signature" or "Owner signature" sections
  - Main applicant or account holder fields
  - Licensee signature fields (holder of a license)
  - Any signature without explicit third-party labels
- **Non-owner signatures** (set to `false` only with clear evidence) typically appear on:
  - Witness signature lines with "Witness" label
  - "Chairman", "Director", or "Authorized Officer" sections with explicit titles
  - Guardian, parent, or representative signature fields with clear labels
  - Co-signer or guarantor sections with "Guarantor" or "Co-signer" labels

## Edge Cases

- **Multiple signatures**: If there are multiple signatures (e.g., witness signatures, co-signers), identify each separately and determine which is the owner's signature
- **Digital signatures**: If you see a digital signature (typed name in script font), note it but mark it as "digital/printed" rather than handwritten
- **Unclear marks**: If you're uncertain whether a mark is a signature, note it with Low confidence
- **Partially visible signatures**: If a signature is cut off or partially obscured, describe what is visible
- **Ambiguous ownership**: If it's unclear whether a signature is from the owner or another party, **default to `true` (owner signature)** unless there is explicit evidence otherwise

## Analysis Approach

1. First, scan for common signature locations
2. Look for handwriting that differs from printed text
3. Identify flowing or stylized writing patterns
4. Check for signature lines or labels
5. Assess each potential signature against the characteristics listed above
6. **Determine signature ownership** by examining:
   - Labels or titles near the signature (e.g., "Owner", "Cardholder", "Witness", "Chairman")
   - Position on the document (primary vs secondary signature locations)
   - Document type and context
   - **When in doubt, default to owner signature (`true`)**
7. Provide clear, actionable results with confidence levels

Remember: Be thorough but conservative. It's better to report Low confidence than to misidentify non-signature elements as signatures.
"""


In [46]:
SIGNATURE_EXTRACTION_PROMPT = """
You are an expert computer vision assistant specializing in document analysis and signature detection. Your task is to identify and locate handwritten signatures within images.

## Task
Analyze the provided image and identify all handwritten signatures present. A signature is a person's name written in a distinctive, personalized handwriting style, typically used for authentication or authorization purposes.

## Instructions

1. **Carefully examine the entire image** for handwritten signatures
2. **Distinguish signatures from other handwritten text** - signatures typically have:
   - A more stylized, flowing, or cursive appearance
   - Personal flourishes or unique characteristics
   - Different ink color or pen pressure than printed text
   - Placement in typical signature locations (bottom of documents, signature lines, etc.)

3. **For each signature found**, provide:
   - **Location**: Describe where in the image the signature appears (e.g., "bottom right corner", "below the printed text", "on the signature line")
   - **Bounding box coordinates**: Estimated x, y, width, height as percentages of image dimensions
     - **IMPORTANT**: Ensure the bounding box is **centered on the signature** with **balanced padding on all sides**
     - Include approximately **10-15% padding** around the actual signature strokes (not too tight, not too loose)
     - The signature should be **visually centered** within the bounding box
     - Adjust the bounding box to maintain symmetry - equal space on left/right and top/bottom where possible
   - **Confidence level**: High, Medium, or Low
   - **Characteristics**: Brief description of the signature's appearance (e.g., "cursive script in blue ink", "printed signature in black")
   - **Legibility**: Whether the signature is legible enough to read the name
   - **Is owner signature**: Boolean indicating if this signature is by the owner/cardholder of the document
     - Set to `true` if the signature appears to be from the document owner, cardholder, Licensee, or primary subject
     - Set to `false` ONLY if there is clear evidence the signature is from an authorized representative, chairman, witness, guarantor, or other third party
     - Look for explicit labels ("Chairman", "Witness", "Authorized Signatory", "Guardian", etc.)
     - **Default to `true` when ownership is ambiguous or unclear**

4. **Exclude the following**:
   - Printed signatures or typed names
   - Initials alone (unless clearly part of a signature)
   - Random handwritten notes that are not signatures
   - Stamps or seal impressions (unless they contain a handwritten component)

## Output Format

Provide your response in the following JSON structure:

```json
{
  "signatures_found": <number>,
  "signatures": [
    {
      "id": 1,
      "location_description": "<descriptive location>",
      "bounding_box": {
        "x": <percentage>,
        "y": <percentage>,
        "width": <percentage>,
        "height": <percentage>
      },
      "confidence": "<High|Medium|Low>",
      "characteristics": "<description>",
      "legible": <true|false>,
      "estimated_name": "<name if legible, otherwise null>",
      "is_owner_signature": <true|false>
    }
  ],
  "analysis_notes": "<any additional observations or challenges>"
}
```

## Examples of What to Look For

**Typical Signature Characteristics:**
- Flowing, connected cursive writing
- Underlines or flourishes beneath the name
- Rapid, confident strokes
- May be partially illegible due to speed of writing
- Often appears on designated signature lines
- May include date nearby

**Common Locations:**
- Bottom of contracts or forms
- Next to "Signature:" or "Signed by:" labels
- On signature lines (indicated by "___________")
- In signature blocks with printed name underneath
- On checks, receipts, or official documents

**Owner vs Non-Owner Signatures:**
- **Owner signatures** typically appear on:
  - Primary signature line of ID cards or documents
  - "Cardholder signature" or "Owner signature" sections
  - Main applicant or account holder fields
  - Licensee signature fields (holder of a license)
  - Any signature without explicit third-party labels
- **Non-owner signatures** (set to `false` only with clear evidence) typically appear on:
  - Witness signature lines with "Witness" label
  - "Chairman", "Director", or "Authorized Officer" sections with explicit titles
  - Guardian, parent, or representative signature fields with clear labels
  - Co-signer or guarantor sections with "Guarantor" or "Co-signer" labels

## Edge Cases

- **Multiple signatures**: If there are multiple signatures (e.g., witness signatures, co-signers), identify each separately and determine which is the owner's signature
- **Digital signatures**: If you see a digital signature (typed name in script font), note it but mark it as "digital/printed" rather than handwritten
- **Unclear marks**: If you're uncertain whether a mark is a signature, note it with Low confidence
- **Partially visible signatures**: If a signature is cut off or partially obscured, describe what is visible
- **Ambiguous ownership**: If it's unclear whether a signature is from the owner or another party, **default to `true` (owner signature)** unless there is explicit evidence otherwise

## Analysis Approach

1. First, scan for common signature locations
2. Look for handwriting that differs from printed text
3. Identify flowing or stylized writing patterns
4. Check for signature lines or labels
5. Assess each potential signature against the characteristics listed above
6. **Determine signature ownership** by examining:
   - Labels or titles near the signature (e.g., "Owner", "Cardholder", "Witness", "Chairman")
   - Position on the document (primary vs secondary signature locations)
   - Document type and context
   - **When in doubt, default to owner signature (`true`)**
7. **Calculate precise bounding boxes**:
   - Identify the exact extent of the signature strokes (leftmost, rightmost, topmost, bottommost points)
   - Add equal padding on all sides (approximately 10-15% of the signature dimensions)
   - Ensure the signature is centered horizontally and vertically within the bounding box
   - Verify the bounding box doesn't include unnecessary whitespace or adjacent elements
8. Provide clear, actionable results with confidence levels

Remember: Be thorough but conservative. It's better to report Low confidence than to misidentify non-signature elements as signatures.

**Bounding Box Quality Checklist:**
- ✓ Signature is centered within the box (not pushed to one edge)
- ✓ Equal padding on left and right sides
- ✓ Equal padding on top and bottom sides
- ✓ Box captures the complete signature including all flourishes
- ✓ Minimal excess whitespace beyond the padding
- ✓ No adjacent text or elements included unless part of the signature
"""


## 3 Specimen Signature Extraction Prompt

This is the detailed prompt that instructs the vision model to extract 3 specimen signatures from images.

In [6]:
SPECIMEN_SIGNATURE_EXTRACTION_PROMPT = """
You are an expert computer vision assistant specializing in document analysis and signature detection. Your task is to identify specimen handwritten signatures in a document and return a SINGLE bounding box that encompasses all specimen signatures.

## Task
Analyze the provided scanned document which contains 3 specimen handwritten signatures. Your goal is to locate all 3 specimen signatures and return ONE bounding box that groups them together.

## Context
- The document is a signature specimen form typically used for bank account opening or verification
- There are exactly 3 handwritten specimen signatures on the document
- The document may also contain a valid ID, but IGNORE it - focus ONLY on the 3 specimen signatures
- The 3 specimen signatures may be arranged horizontally, vertically, or in a grid pattern

## Instructions

1. **Locate all 3 specimen handwritten signatures** in the document
   - Specimen signatures typically have:
     - A more stylized, flowing, or cursive appearance
     - Personal flourishes or unique characteristics
     - Placement on designated signature lines or boxes
     - Labels like "Specimen Signature 1", "Signature", or numbered signature fields

2. **Identify the region containing all 3 signatures**:
   - Find the leftmost point of all 3 signatures
   - Find the rightmost point of all 3 signatures
   - Find the topmost point of all 3 signatures
   - Find the bottommost point of all 3 signatures

3. **Create ONE bounding box that encompasses all 3 signatures**:
   - **x**: The left edge of the bounding box (percentage from left of image)
   - **y**: The top edge of the bounding box (percentage from top of image)
   - **width**: The width of the bounding box (percentage of image width)
   - **height**: The height of the bounding box (percentage of image height)
   - **IMPORTANT**: Include approximately **5-10% padding** around the entire group
   - The bounding box should capture ALL 3 signatures as a single group
   - Ensure no signature is cut off or excluded

4. **CRITICAL RULES**:
   - Return ONLY ONE signature entry in the signatures array
   - The bounding box must contain ALL 3 specimen signatures grouped together
   - IGNORE the valid ID section - do not include it in the bounding box
   - IGNORE printed text, labels, or form elements outside the signature area
   - Focus on the handwritten signature strokes only

## Output Format

Provide your response in the following JSON structure:

```json
{
  "signatures_found": 1,
  "signatures": [
    {
      "id": 1,
      "location_description": "<descriptive location of the grouped specimen signature area>",
      "bounding_box": {
        "x": <percentage>,
        "y": <percentage>,
        "width": <percentage>,
        "height": <percentage>
      },
      "confidence": "<High|Medium|Low>",
      "characteristics": "<description of the 3 specimen signatures and their arrangement>",
      "legible": <true|false>,
      "estimated_name": "<name if legible, otherwise null>"
    }
  ],
  "analysis_notes": "<observations about the 3 specimen signatures, their arrangement (horizontal/vertical/grid), and grouping>"
}
```

## Field Descriptions

- **signatures_found**: Must be 1 (representing the single grouped bounding box)
- **signatures**: Array with ONE entry containing the grouped bounding box
  - **id**: Always 1
  - **location_description**: Describe where the specimen signature area is located (e.g., "center of document, below the form header")
  - **bounding_box**: Single box containing all 3 specimen signatures
    - **x**: Left edge as percentage (0-100)
    - **y**: Top edge as percentage (0-100)
    - **width**: Width as percentage (0-100)
    - **height**: Height as percentage (0-100)
  - **confidence**: Your confidence in detecting all 3 signatures (High/Medium/Low)
  - **characteristics**: Describe the 3 signatures and their arrangement (e.g., "Three specimen signatures arranged horizontally in blue ink")
  - **legible**: Whether any of the signatures are legible
  - **estimated_name**: The name if any signature is legible, otherwise null
- **analysis_notes**: Include arrangement type (horizontal/vertical/grid/mixed) and any observations

## Examples of Signature Arrangements

1. **Horizontal**: Three signatures side-by-side in a row
2. **Vertical**: Three signatures stacked on top of each other
3. **Grid**: Signatures arranged in a 2x2 or 3x1 grid pattern
4. **Mixed**: Signatures arranged irregularly

## Analysis Approach

1. Scan the entire document for handwritten signatures (ignore printed elements)
2. Identify which signatures are specimen signatures (typically 3 on the form)
3. Ignore any valid ID images or text sections
4. Determine the spatial arrangement of the 3 specimen signatures
5. Calculate the bounding box that encompasses all 3:
   - Find min_x (leftmost signature edge)
   - Find max_x (rightmost signature edge)
   - Find min_y (topmost signature edge)
   - Find max_y (bottommost signature edge)
   - Add 5-10% padding to create the final grouped bounding box
6. Verify all 3 signatures are within the box
7. Return as a single entry in the signatures array

## Quality Checklist

- ✓ All 3 specimen signatures are fully contained within the bounding box
- ✓ No signatures are cut off or partially excluded
- ✓ Valid ID section is NOT included in the bounding box
- ✓ Reasonable padding (5-10%) around the signature group
- ✓ Bounding box is as tight as possible while containing all signatures
- ✓ Only ONE entry in the signatures array (signatures_found = 1)
- ✓ Characteristics field describes all 3 signatures and their arrangement

Remember: The goal is to return a SINGLE bounding box that groups all 3 specimen signatures together, formatted as one entry in the original JSON structure.
"""

## Helper Functions

In [36]:
def encode_image_to_base64(image_path: str) -> str:
    """Encode an image file to base64 string."""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')


def extract_signatures(image_path: str, model: str = "gpt-4.1") -> dict[str, Any]:
    """
    Extract handwritten signatures from an image using Azure OpenAI Vision API.
    
    Args:
        image_path: Path to the image file
        model: Azure OpenAI model deployment name (must support vision)
        
    Returns:
        Dictionary containing signature extraction results
    """
    # Encode image to base64
    base64_image = encode_image_to_base64(image_path)
    
    # Make API call
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "You are an expert at analyzing documents and detecting handwritten signatures. Always respond with valid JSON."
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": SIGNATURE_EXTRACTION_PROMPT
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{base64_image}",
                            "detail": "high"
                        }
                    }
                ]
            }
        ],
        temperature=0.0,  # Use deterministic output for consistency
        response_format={
            "type": "json_object"
        },
        timeout=60
    )
    
    # Extract and parse response
    content = response.choices[0].message.content
    print(content)
    
    # Handle None content
    if content is None:
        return {"signatures_found": 0, "signatures": [], "analysis_notes": "No content returned from API"}
    
    # Try to extract JSON from response (in case it's wrapped in markdown)
    if "```json" in content:
        json_start = content.find("```json") + 7
        json_end = content.find("```", json_start)
        content = content[json_start:json_end].strip()
    elif "```" in content:
        json_start = content.find("```") + 3
        json_end = content.find("```", json_start)
        content = content[json_start:json_end].strip()
    
    result: dict[str, Any] = json.loads(content)
    return result

In [40]:
def save_cropped_signatures(
    image_path: str,
    signatures: list[dict[str, Any]],
    output_dir: str = "../tmp"
) -> list[str]:
    """
    Crop and save signature regions from an image based on bounding box coordinates.
    Only saves signatures where is_owner_signature is True.
    
    SERVERLESS COMPATIBLE: File saving can be disabled by not calling this function.
    All processing uses in-memory operations via PIL/OpenCV.
    
    Args:
        image_path: Path to the original image file
        signatures: List of signature dictionaries with bounding_box (x, y, width, height in percentages)
        output_dir: Directory to save cropped images (default: "../tmp")
    
    Returns:
        List of saved file paths
    """
    if not signatures:
        return []
    
    # Filter signatures to only include owner signatures
    owner_signatures = [sig for sig in signatures if sig.get('is_owner_signature', False)]
    
    if not owner_signatures:
        print("ℹ No owner signatures found to save")
        return []
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Load the original image
    image = Image.open(image_path).convert("RGBA")
    img_width, img_height = image.size
    
    saved_files: list[str] = []
    timestamp = str(int(time.time() * 1000))
    file_stem = Path(image_path).stem
    
    for sig in owner_signatures:
        sig_id = sig.get('id', 0)
        bbox = sig.get('bounding_box', {})
        
        # Convert percentage-based coordinates to pixel coordinates
        x_pct = bbox.get('x', 0)
        y_pct = bbox.get('y', 0)
        width_pct = bbox.get('width', 0)
        height_pct = bbox.get('height', 0)
        
        # Calculate pixel coordinates
        x1 = int((x_pct / 100) * img_width)
        y1 = int((y_pct / 100) * img_height)
        x2 = int(((x_pct + width_pct) / 100) * img_width)
        y2 = int(((y_pct + height_pct) / 100) * img_height)
        
        # Add padding
        padding = 100
        x1 = max(0, x1 - padding)
        y1 = max(0, y1 - padding)
        x2 = min(img_width, x2 + padding)
        y2 = min(img_height, y2 + padding)
        
        # Crop the signature region
        cropped = image.crop((x1, y1, x2, y2))
        
        # Save cropped signature
        filename = f"{file_stem}_sig_{sig_id}_{timestamp}.png"
        output_path = Path(output_dir) / filename
        cropped.save(output_path, format='PNG')
        
        saved_files.append(str(output_path))
        print(f"✓ Saved owner signature: {output_path}")
    
    return saved_files


## Example Usage

In [14]:
# Example: Process a single image
# image_path = "../testdata/valid-id/testing/valid_id1.png"
image_path = "../tmp/upscaled.png"

# Check if file exists
if Path(image_path).exists():
    print(f"Analyzing image: {image_path}")
    result = extract_signatures(image_path)
    
    # Pretty print results
    print("\n" + "="*50)
    print("SIGNATURE EXTRACTION RESULTS")
    print("="*50)
    print(f"\nSignatures found: {result['signatures_found']}")
    print(f"\nAnalysis notes: {result.get('analysis_notes', 'N/A')}")
    
    if result['signatures_found'] > 0:
        saved_files = save_cropped_signatures(image_path, result['signatures'])
        print(f"Saved {len(saved_files)} signature crops")
        print("\n" + "-"*50)
        for sig in result['signatures']:
            print(f"\nSignature #{sig['id']}:")
            print(f"  Location: {sig['location_description']}")
            print(f"  Confidence: {sig['confidence']}")
            print(f"  Characteristics: {sig['characteristics']}")
            print(f"  Legible: {sig['legible']}")
            if sig.get('estimated_name'):
                print(f"  Estimated name: {sig['estimated_name']}")
            print(f"  Bounding box: {sig['bounding_box']}")
else:
    print(f"Image not found: {image_path}")
    print("\nPlease update the image_path variable with a valid path to test the signature extraction.")

Analyzing image: ../tmp/upscaled.png
{
  "signatures_found": 1,
  "signatures": [
    {
      "id": 1,
      "location_description": "center of the document, directly above the printed city name and below the birthdate",
      "bounding_box": {
        "x": 36,
        "y": 44,
        "width": 34,
        "height": 13
      },
      "confidence": "High",
      "characteristics": "stylized cursive script in black ink, flowing strokes with personal flourishes, distinct from printed text",
      "legible": true,
      "estimated_name": "Bayot",
      "is_owner_signature": true
    }
  ],
  "analysis_notes": "The signature is clearly handwritten, positioned in a typical signature location on an identification card, and visually distinct from the printed text. No explicit third-party labels are present, so ownership is attributed to the cardholder. The bounding box is centered with balanced padding, capturing the full extent of the signature strokes."
}

SIGNATURE EXTRACTION RESULTS

Signa

## Batch Processing Example

In [44]:
def process_directory(
    directory_path: str = "../testdata/valid-id/testing",
    output_dir: str = "../tmp",
    model: str = "gpt-4.1"
) -> dict[str, Any]:
    """
    Process all images in a directory and extract signatures.
    
    SERVERLESS COMPATIBLE: All processing is done in-memory by default.
    File saving controlled by save_cropped_signatures() function call.
    
    Args:
        directory_path: Path to directory containing images (default: "../testdata/valid-id/testing")
        output_dir: Directory to save cropped images (default: "../tmp")
        model: Azure OpenAI model deployment name (default: "gpt-4.1")
    
    Returns:
        Dictionary mapping filenames to extraction results
    """
    results: dict[str, Any] = {}
    supported_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tiff"}
    
    directory = Path(directory_path)
    if not directory.exists():
        raise FileNotFoundError(f"Directory not found: {directory_path}")
    
    files = [f for f in directory.iterdir() if f.suffix.lower() in supported_extensions]
    
    print(f"\nFound {len(files)} file(s) to process\n")
    print("="*60)
    
    for idx, file in enumerate(files, 1):
        print(f"\n[{idx}/{len(files)}] Processing: {file.name}")
        print("-"*60)
        
        try:
            result = extract_signatures(str(file), model=model)
            results[file.name] = result
            
            # Optionally save cropped signatures
            if result['signatures_found'] > 0:
                saved_files = save_cropped_signatures(str(file), result['signatures'], output_dir)
                print(f"✓ Success: {result['signatures_found']} signature(s) found, {len(saved_files)} crops saved")
            else:
                print(f"✓ Success: {result['signatures_found']} signature(s) found")
        except Exception as e:
            print(f"✗ Error: {str(e)}")
            results[file.name] = {
                "file": str(file),
                "error": str(e),
                "signatures_found": 0
            }
    
    print("\n" + "="*60)
    print("BATCH PROCESSING COMPLETE")
    print("="*60)
    
    total_signatures = sum(
        r.get('signatures_found', 0) for r in results.values()
    )
    successful = sum(
        1 for r in results.values() if 'error' not in r
    )
    
    print(f"\nProcessed: {len(files)} file(s)")
    print(f"Successful: {successful}")
    print(f"Failed: {len(files) - successful}")
    print(f"Total signatures extracted: {total_signatures}")
    
    return results


# Example: Process a directory (uncomment to run)
batch_results = process_directory("../tmp/s0-nocenter-100/UPSCALED", "../tmp/s0-nocenter-100/GPT-100")


Found 7 file(s) to process


[1/7] Processing: valid_id1_sig_1_1763187513397_sharp_gray1_upscaled_x2.png
------------------------------------------------------------
{
  "signatures_found": 1,
  "signatures": [
    {
      "id": 1,
      "location_description": "center-left area, below the printed text and above the address line",
      "bounding_box": {
        "x": 28,
        "y": 44,
        "width": 38,
        "height": 13
      },
      "confidence": "High",
      "characteristics": "stylized, flowing cursive script in dark ink, with distinctive personal flourishes and rapid strokes",
      "legible": false,
      "estimated_name": null,
      "is_owner_signature": true
    }
  ],
  "analysis_notes": "The detected signature is in a typical location for cardholder signatures on identification documents. It is clearly handwritten and distinct from the printed text. No explicit labels indicating a third-party signer are present, so ownership is attributed to the document holder by

## Visualization Helper (Optional)

Draw bounding boxes on the image to visualize detected signatures.

In [ ]:
# Optional: Install PIL/Pillow for visualization
# !pip install pillow

from PIL import Image, ImageDraw

def visualize_signatures(image_path: str, result: dict[str, Any], output_path: str | None = None):
    """
    Draw bounding boxes on image for detected signatures.
    
    Args:
        image_path: Path to original image
        result: Signature extraction result dictionary
        output_path: Path to save annotated image (optional)
    """
    # Open image
    img = Image.open(image_path)
    draw = ImageDraw.Draw(img)
    
    # Get image dimensions
    width, height = img.size
    
    # Draw bounding boxes for each signature
    for sig in result.get('signatures', []):
        bbox = sig['bounding_box']
        
        # Convert percentage to pixels
        x = int(bbox['x'] * width / 100)
        y = int(bbox['y'] * height / 100)
        w = int(bbox['width'] * width / 100)
        h = int(bbox['height'] * height / 100)
        
        # Choose color based on confidence
        color = {
            'High': 'green',
            'Medium': 'orange',
            'Low': 'red'
        }.get(sig['confidence'], 'blue')
        
        # Draw rectangle
        draw.rectangle([x, y, x+w, y+h], outline=color, width=3)
        
        # Add label
        label = f"Sig #{sig['id']} ({sig['confidence']})"
        draw.text((x, y-20), label, fill=color)
    
    # Display or save
    if output_path:
        img.save(output_path)
        print(f"Annotated image saved to: {output_path}")
    
    return img


# Example usage (uncomment to run)
# if Path(image_path).exists():
#     annotated_img = visualize_signatures(image_path, result, "annotated_output.jpg")
#     annotated_img.show()

## Notes

**Requirements:**
- Azure OpenAI account with a vision-capable model deployed (e.g., GPT-4o, GPT-4-vision)
- Environment variables configured in `.env` file:
  - `AZURE_OPENAI_API_KEY`
  - `AZURE_OPENAI_ENDPOINT`
  - `AZURE_OPENAI_API_VERSION`

**Tips:**
- Use high-resolution images for best results
- The `detail: "high"` parameter enables more detailed image analysis
- Adjust `max_tokens` based on expected number of signatures
- Temperature=0 ensures consistent outputs
- For production use, add retry logic and error handling

**Performance Considerations:**
- Vision API calls are more expensive than text-only calls
- Processing time depends on image size and complexity
- Consider batching requests for large volumes
- Cache results when processing the same image multiple times